# Q-Learning Training Pipeline with TD Error Prioritization

## Overview
This notebook implements an efficient Q-learning training pipeline for Connect 4 that pre-computes TD errors during data generation and uses them for prioritized experience replay.

---

## Dependencies and Helper Functions

### External Dependencies
- `torch`, `torch.nn`, `torch.optim` - PyTorch for neural network operations
- `numpy` - Numerical operations
- `BoardProcessor` - Handles game state encoding/decoding (from `board_processor.py`)
- `FeatureGenerator` - Generates convolutional features from board states (from `feature_generator.py`)

### Required Helper Functions (from previous cells)

#### `get_current_q(state_action_features, online_model, scaler, player)`
- **Purpose**: Calculate current Q-value for a state-action pair
- **Inputs**:
  - `state_action_features`: Concatenated features of current and next state
  - `online_model`: Neural network for Q-value estimation
  - `scaler`: Feature scaler for normalization
  - `player`: Current player (1 or -1)
- **Returns**: Q-value adjusted for player perspective

#### `calculate_target_q(moves, position_i, player, online_model, target_model, scaler, feature_gen, gamma)`
- **Purpose**: Calculate target Q-value using Double DQN logic
- **Inputs**:
  - `moves`: Complete game move sequence
  - `position_i`: Current position in game
  - `player`: Current player
  - `online_model`: Network for action selection
  - `target_model`: Network for action evaluation
  - `scaler`, `feature_gen`: Feature processing utilities
  - `gamma`: Discount factor
- **Returns**: Target Q-value for TD update
- **Logic**: Handles terminal states (wins/draws) and uses Double DQN for non-terminal states

---

## Main Components

### 1. Data Structures

#### `TrainingTuple` (NamedTuple)
```
Fields:
- state_action: Feature vector for state-action pair
- target_q: Target Q-value after TD update
- td_error: Absolute TD error for prioritization
- move_number: Position in game (for endgame bonuses)
- game_length: Total game length (for close game detection)
```

### 2. Data Generation

#### `generate_training_tuples_with_td(...)`
- **Purpose**: Generate training data with pre-computed TD errors
- **Dependencies**: `get_current_q`, `calculate_target_q`
- **Process**:
  1. Decode each game into move sequences
  2. For each non-terminal position:
     - Extract state features
     - Calculate current Q-value
     - Calculate target Q-value (Double DQN)
     - Compute TD error = target - current
     - Apply TD update: new_q = current + α * TD_error
     - Store as TrainingTuple with metadata
- **Returns**: List of TrainingTuples with all computations done upfront

### 3. Replay Buffer

#### `SmartReplayBuffer` (Class)
- **Purpose**: Efficient storage and sampling of training experiences

##### Methods:
- **`__init__(capacity)`**: Initialize circular buffer
- **`push(tuple_data)`**: Add new experience
- **`sample_prioritized(...)`**:
  - Sample based on TD error magnitude
  - Apply endgame bonuses (moves 35+)
  - Apply close game bonuses (40+ moves)
  - Calculate importance sampling weights
  - Returns: (states, targets, weights, td_errors, indices)
- **`sample_uniform(batch_size)`**: Simple random sampling baseline
- **`get_statistics()`**: Buffer analytics (avg TD error, endgame ratio, etc.)

### 4. Training Functions

#### `train_with_smart_buffer(...)`
- **Purpose**: Sophisticated training with prioritized replay
- **Dependencies**: `SmartReplayBuffer` with pre-filled data
- **Key Features**:
  - **Optimizer**: AdamW with weight decay (more stable than Adam)
  - **Loss**: SmoothL1Loss (Huber) - robust to Q-value outliers
  - **LR Schedule**: Cosine annealing (smooth decay)
  - **Beta annealing**: 0.4 → 1.0 for importance sampling
  - **Gradient clipping**: Prevents instability
- **Process**:
  1. Sample batch (prioritized or uniform)
  2. Forward pass with scaled features
  3. Calculate weighted loss
  4. Backprop with gradient clipping
  5. Update learning rate per epoch
- **Returns**: (loss_history, td_error_history)

#### `minimal_train_with_td(...)`
- **Purpose**: Simplified training focusing on high TD error samples
- **Dependencies**: Pre-computed TrainingTuples
- **Key Features**:
  - **Optimizer**: SGD with momentum (simpler, more predictable)
  - **Strategy**: Sort by TD error, train on top 50%
  - **No bells & whistles**: Good for debugging/baselines
- **Process**:
  1. Sort tuples by TD error
  2. Train on highest error samples
  3. Simple MSE loss
  4. Basic SGD updates

#### `full_training_pipeline(...)`
- **Purpose**: Complete end-to-end training workflow
- **Dependencies**: All above functions
- **Process**:
  1. Load pretrained model and scaler
  2. Generate training tuples with TD errors
  3. Fill replay buffer
  4. Train with prioritized replay
  5. Save improved model
- **Returns**: (trained_model, losses, td_history)

---

## Key Design Decisions

### Why Pre-compute TD Errors?
- **Efficiency**: Calculate once, use many times
- **Flexibility**: Can experiment with prioritization without model calls
- **Metadata**: Store game context (move number, game length) for smarter sampling

### Why SmoothL1Loss (Huber)?
- Q-learning can have large initial errors
- MSE would cause gradient explosion
- Huber is quadratic for small errors (stable) but linear for large errors (robust)

### Why Double DQN?
- Reduces overestimation bias in Q-values
- Online network selects actions
- Target network evaluates them
- More stable learning

### Why Prioritized Replay?
- Focus on surprising/important transitions
- Endgame positions (35+ moves) are crucial in Connect 4
- Close games (40+ moves) contain valuable strategic information

### Why Beta Annealing?
- Early training: Aggressive prioritization helps learn faster
- Late training: Need unbiased sampling for convergence
- Smooth transition from 0.4 to 1.0

---

## Usage Example

```python
# 1. Load your game codes
with open('game_codes.txt', 'r') as f:
    game_codes = [line.strip() for line in f.readlines()[:1000]]

# 2. Generate training data
tuples = generate_training_tuples_with_td(
    game_codes, online_model, target_model,
    scaler, feature_gen, alpha=0.1, gamma=0.99
)

# 3. Fill replay buffer
buffer = SmartReplayBuffer(capacity=100000)
for t in tuples:
    buffer.push(t)

# 4. Train
losses, td_hist = train_with_smart_buffer(
    online_model, buffer, scaler,
    epochs=100, use_prioritized=True
)
```

---

## Hyperparameter Recommendations for Connect 4

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `alpha` (TD learning rate) | 0.1 | Conservative updates for stability |
| `gamma` (discount) | 0.99 | Future positions matter in Connect 4 |
| `lr` (network learning rate) | 5e-4 → 5e-5 | Start higher, decay smoothly |
| `batch_size` | 256 | Large batches for stable gradients |
| `endgame_threshold` | 35 moves | When board gets strategically complex |
| `endgame_bonus` | 2.0x | Critical positions deserve more attention |
| `gradient_clip` | 10.0 | Prevent instability without over-constraining |
| `beta` (IS correction) | 0.4 → 1.0 | Gradual transition to unbiased sampling |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import namedtuple
import heapq
import matplotlib.pyplot as plt

# Named tuple for clarity
TrainingTuple = namedtuple('TrainingTuple', ['state_action', 'target_q', 'td_error', 'move_number', 'game_length'])

In [ ]:
from board_processor import BoardProcessor
from feature_generator import FeatureGenerator
import os

class QNetwork(nn.Module):
    def __init__(self, input_dim=138):
        super().__init__()
        layers = []
        for h in [256, 128, 64, 32, 16, 8]:
            layers.extend([nn.Linear(input_dim, h), nn.Tanh()])
            input_dim = h
        layers.extend([nn.Linear(h, 1), nn.Tanh()])
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)

In [ ]:

def get_current_q(state_action_features, online_model, scaler, player):
    """Get current Q-value prediction from online model"""
    scaled = scaler.transform([state_action_features])
    with torch.no_grad():
        q = online_model(torch.FloatTensor(scaled).to(online_model.device if hasattr(online_model, 'device') else 'cpu')).item()
    return q * player  # Adjust for player perspective

def calculate_target_q(moves, position_i, player, online_model, target_model, scaler, feature_gen, gamma=0.99):
    """Calculate target Q-value using Double DQN logic"""
    # Check if game ends after our move (position i+1)
    board_after_our_move = BoardProcessor()
    board_after_our_move.generate_state_list(moves[:position_i+1])
    _, feats_after_our_move = feature_gen.convolution_feature_gen(board_after_our_move.state_list)

    # Terminal after our move?
    if 4 in feats_after_our_move:
        return 1 * player  # We win
    elif -4 in feats_after_our_move:
        return -1 * player  # We lose (shouldn't happen)
    elif position_i + 1 >= len(moves):
        return 0  # Draw

    # Check if game ends after opponent's move (position i+2)
    if position_i + 3 >= 42:
        # print("^"*15 + f"Debug - IT GOT triggered when {position_i} plus two is greater than 42")
        return 0  # Draw
    # else: print("^"*15 + f"Debug - ever gets triggered when {position_i} plus two is greater than 42")

    board_after_opp = BoardProcessor()
    board_after_opp.generate_state_list(moves[:position_i+2])
    _, feats_after_opp = feature_gen.convolution_feature_gen(board_after_opp.state_list)

    if 4 in feats_after_opp or -4 in feats_after_opp:
        return -1   # Opponent wins

    # Non-terminal: calculate Q-value of next state
    next_board = BoardProcessor()
    next_board.generate_state_list(moves[:position_i+2])
    _, next_curr_feats = feature_gen.convolution_feature_gen(next_board.state_list)

    # Get Q-values for all possible next moves using ONLINE network for selection
    online_q_values = []
    for col in range(7):
        if len(next_board.state_list[col]) < 6:  # Legal move
            next_state = [c[:] for c in next_board.state_list]
            next_state[col].append(player)  # Same player's turn
            _, next_feats = feature_gen.convolution_feature_gen(next_state)

            # Check for immediate win
            if 4 * player in next_feats:
                online_q_values.append((col, 1.0))
            else:
                # Get Q-value from ONLINE model
                features = np.concatenate([next_curr_feats, next_feats])
                scaled = scaler.transform([features])
                with torch.no_grad():
                    q = online_model(torch.FloatTensor(scaled).to(online_model.device if hasattr(online_model, 'device') else 'cpu')).item() * player
                    online_q_values.append((col, q))

    if not online_q_values:
        return 0  # No legal moves = draw

    # DOUBLE DQN: Online network selects best action
    best_action = max(online_q_values, key=lambda x: x[1])[0]

    # TARGET network evaluates the selected action
    best_next_state = [c[:] for c in next_board.state_list]
    best_next_state[best_action].append(player)
    _, best_next_feats = feature_gen.convolution_feature_gen(best_next_state)

    # Check for immediate win with selected action
    if 4 * player in best_next_feats:
        target_q_value = 1.0
    else:
        # Evaluate using TARGET network
        best_features = np.concatenate([next_curr_feats, best_next_feats])
        best_scaled = scaler.transform([best_features])
        with torch.no_grad():
            target_q_value = target_model(torch.FloatTensor(best_scaled).to(target_model.device if hasattr(target_model, 'device') else 'cpu')).item() * player

    return gamma * target_q_value

In [ ]:
#Most advanced so far
def generate_training_tuples_with_td(game_codes, online_model, target_model, scaler,
                                     feature_gen,
                                     alpha=0.1, gamma=0.99, max_tuples=None):
    """
    Generate (state_action_features, target_q, td_error, metadata) tuples from game codes
    TD error is calculated during generation to avoid redundant forward passes
    """
    training_tuples = []

    for game_idx, game_code in enumerate(game_codes):
        board = BoardProcessor()
        moves = board.decode_moves_code(game_code)
        board.generate_state_list(moves)
        game_length = len(moves)

        # Process each non-terminal position
        for i in range(len(moves) - 1):  # Skip final position
            # Current state and player
            temp_board = BoardProcessor()
            temp_board.generate_state_list(moves[:i])
            player = 1 if (i % 2) == 0 else -1

            # Get current state features
            _, curr_feats = feature_gen.convolution_feature_gen(temp_board.state_list)

            # Action taken and resulting state
            action = moves[i]
            next_state = [col[:] for col in temp_board.state_list]
            next_state[action].append(player)
            _, next_feats = feature_gen.convolution_feature_gen(next_state)

            # Create state-action input features
            state_action_features = np.concatenate([curr_feats, next_feats])

            # Get CURRENT Q-value (we're computing this anyway!)
            current_q = get_current_q(state_action_features, online_model, scaler, player)

            # Calculate target Q-value using Double DQN
            target_q_raw = calculate_target_q(moves, i, player, online_model, target_model,
                                             scaler, feature_gen, gamma)

            # TD error (before applying alpha)
            td_error = target_q_raw - current_q

            # New Q value after TD update
            new_q = current_q + alpha * td_error

            # Store tuple with all information
            tuple_data = TrainingTuple(
                state_action=state_action_features,
                target_q=new_q,
                td_error=abs(td_error),  # Store absolute TD error for prioritization
                move_number=i,
                game_length=game_length
            )
            training_tuples.append(tuple_data)

            # Early exit if we have enough tuples
            if max_tuples and len(training_tuples) >= max_tuples:
                return training_tuples

    return training_tuples

In [ ]:

# This one is smart - can prioritize samples into batches
class SmartReplayBuffer:
    """
    Replay buffer that uses pre-computed TD errors and game metadata
    """
    def __init__(self, capacity=50000):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def push(self, tuple_data):
        """Add a training tuple to the buffer"""
        if len(self.buffer) < self.capacity:
            self.buffer.append(tuple_data)
        else:
            self.buffer[self.position] = tuple_data
        self.position = (self.position + 1) % self.capacity

    def sample_prioritized(self, batch_size, alpha=0.6, beta=0.4,
                          endgame_bonus=2.0, endgame_threshold=35):
        """
        Sample using pre-computed TD errors with endgame position bonus

        Args:
            alpha: Priority exponent for TD errors
            beta: Importance sampling correction
            endgame_bonus: Multiplier for endgame positions
            endgame_threshold: Positions after this move get bonus
        """
        if len(self.buffer) < batch_size:
            return None

        # Calculate priorities using stored TD errors
        priorities = []
        for tuple_data in self.buffer:
            priority = (tuple_data.td_error + 1e-6) ** alpha

            # Bonus for endgame positions
            if tuple_data.move_number >= endgame_threshold:
                priority *= endgame_bonus

            # Additional bonus for very close games
            if tuple_data.game_length >= 40:  # Near-draw games
                priority *= 1.5

            priorities.append(priority)

        # Convert to probabilities
        priorities = np.array(priorities)
        probs = priorities / priorities.sum()

        # Sample indices
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)

        # Calculate importance sampling weights
        weights = (len(self.buffer) * probs[indices]) ** (-beta)
        weights /= weights.max()

        # Create batch
        batch = [self.buffer[idx] for idx in indices]

        # states = torch.FloatTensor([t.state_action for t in batch])
        states = torch.FloatTensor(np.array([t.state_action for t in batch]))
        targets = torch.FloatTensor([t.target_q for t in batch])
        weights = torch.FloatTensor(weights)
        td_errors = torch.FloatTensor([t.td_error for t in batch])

        return states, targets, weights, td_errors, indices

    def sample_uniform(self, batch_size):
        """Simple uniform sampling for comparison"""
        if len(self.buffer) < batch_size:
            return None

        indices = np.random.choice(len(self.buffer), batch_size)
        batch = [self.buffer[idx] for idx in indices]

        # states = torch.FloatTensor([t.state_action for t in batch])
        states = torch.FloatTensor(np.array([t.state_action for t in batch]))
        targets = torch.FloatTensor([t.target_q for t in batch])

        return states, targets

    def plot_td_error_analysis(self, figsize=(15, 5), alpha=0.6, s=20):
        """
        Plot TD error analysis with three scatter plots:
        1. TD Error vs Move Number
        2. TD Error vs Game Length
        3. TD Error vs Endgame Ratio

        Works with existing TrainingTuple: ['state_action', 'target_q', 'td_error', 'move_number', 'game_length']
        """
        import matplotlib.pyplot as plt
        import numpy as np

        if not self.buffer:
            print("Buffer is empty! Add some training examples first.")
            return

        # Extract data from TrainingTuples
        td_errors = [t.td_error for t in self.buffer]
        move_numbers = [t.move_number for t in self.buffer]
        game_lengths = [t.game_length for t in self.buffer]
        # Calculate endgame ratio from existing fields
        endgame_ratios = [move_num / max(game_len - 1, 1) for move_num, game_len in zip(move_numbers, game_lengths)]

        # Create three subplots
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=figsize)

        # Plot 1: TD Error vs Move Number
        ax1.scatter(move_numbers, td_errors, alpha=alpha, s=s, color='blue')
        ax1.set_xlabel('Move Number')
        ax1.set_ylabel('TD Error')
        ax1.set_title('TD Error vs Move Number')
        ax1.grid(True, alpha=0.3)
        ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5)

        # Plot 2: TD Error vs Game Length
        ax2.scatter(game_lengths, td_errors, alpha=alpha, s=s, color='green')
        ax2.set_xlabel('Game Length (total moves)')
        ax2.set_ylabel('TD Error')
        ax2.set_title('TD Error vs Game Length')
        ax2.grid(True, alpha=0.3)
        ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)

        # Plot 3: TD Error vs Endgame Ratio
        ax3.scatter(endgame_ratios, td_errors, alpha=alpha, s=s, color='red')
        ax3.set_xlabel('Endgame Ratio (move/total_moves)')
        ax3.set_ylabel('TD Error')
        ax3.set_title('TD Error vs Endgame Ratio')
        ax3.grid(True, alpha=0.3)
        ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

        # Print some statistics
        print(f"\nTD Error Analysis Statistics:")
        print(f"Total examples: {len(self.buffer)}")
        print(f"TD Error - Mean: {np.mean(td_errors):.4f}, Std: {np.std(td_errors):.4f}")
        print(f"TD Error - Min: {np.min(td_errors):.4f}, Max: {np.max(td_errors):.4f}")
        print(f"Move numbers - Range: {np.min(move_numbers)} to {np.max(move_numbers)}")
        print(f"Game lengths - Range: {np.min(game_lengths)} to {np.max(game_lengths)}")
        print(f"Endgame ratios - Range: {np.min(endgame_ratios):.3f} to {np.max(endgame_ratios):.3f}")

    def get_statistics(self):
        """Get buffer statistics for monitoring"""
        if not self.buffer:
            return {}

        td_errors = [t.td_error for t in self.buffer]
        move_numbers = [t.move_number for t in self.buffer]
        game_lengths = [t.game_length for t in self.buffer]

        return {
            'size': len(self.buffer),
            'avg_td_error': np.mean(td_errors),
            'max_td_error': np.max(td_errors),
            'min_td_error': np.min(td_errors),
            'avg_move_number': np.mean(move_numbers),
            'avg_game_length': np.mean(game_lengths),
            'endgame_ratio': sum(1 for m in move_numbers if m >= 35) / len(move_numbers)
        }

In [ ]:

#More complex - trains with smart buffer
def train_with_smart_buffer(online_model, replay_buffer, scaler,
                           epochs=100, batch_size=256, lr=5e-4,
                           use_prioritized=True):
    """
    Simplified training using pre-computed TD errors
    """
    device = next(online_model.parameters()).device

    # Optimizer choice
    optimizer = optim.AdamW(  # AdamW often better than Adam for RL
        online_model.parameters(),
        lr=lr,
        weight_decay=1e-5,
        amsgrad=True  # More stable variant
    )

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr/10
    )

    criterion = nn.SmoothL1Loss(reduction='none')

    losses = []
    td_errors_history = []

    for epoch in range(epochs):
        epoch_loss = 0
        epoch_td = 0
        num_batches = 10  # Fixed batches per epoch

        for _ in range(num_batches):
            if use_prioritized:
                # Anneal beta from 0.4 to 1.0
                beta = min(1.0, 0.4 + (epoch / epochs) * 0.6)
                batch = replay_buffer.sample_prioritized(
                    batch_size, alpha=0.6, beta=beta
                )
                if batch is None:
                    continue
                states, targets, weights, td_errors, _ = batch
                weights = weights.to(device)
            else:
                batch = replay_buffer.sample_uniform(batch_size)
                if batch is None:
                    continue
                states, targets = batch
                weights = torch.ones(batch_size).to(device)
                td_errors = torch.zeros(batch_size)

            states = states.to(device)
            targets = targets.to(device)

            # Scale features
            states_scaled = scaler.transform(states.cpu().numpy())
            states_scaled = torch.FloatTensor(states_scaled).to(device)

            # Forward pass
            predictions = online_model(states_scaled).squeeze()

            # Weighted loss
            losses_batch = criterion(predictions, targets)
            loss = (losses_batch * weights).mean()

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(online_model.parameters(), 10)

            optimizer.step()

            epoch_loss += loss.item()
            epoch_td += td_errors.mean().item()

        # Step scheduler
        scheduler.step()

        # Log progress
        avg_loss = epoch_loss / num_batches
        avg_td = epoch_td / num_batches
        losses.append(avg_loss)
        td_errors_history.append(avg_td)

        if epoch % 10 == 0:
            stats = replay_buffer.get_statistics()
            print(f"Epoch {epoch:3d}: Loss={avg_loss:.6f}, "
                  f"AvgTD={avg_td:.4f}, BufferTD={stats['avg_td_error']:.4f}, "
                  f"LR={scheduler.get_last_lr()[0]:.6f}")

    return losses, td_errors_history

In [ ]:
# Cell that disassembles full_training_pipeline
# This particular cell takes 6000 game codes and gets ready to produce TD difference cells.
codes_file = os.path.expanduser('~/Downloads/replayMem.txt')
skip_rows = 1e6
skip_rows = int(skip_rows)
game_codes = []
with open(codes_file, 'r') as f:
    print(f"Skipping a {skip_rows} rows")
    for _ in range(skip_rows):
        f.readline()
    for i, line in enumerate(f):
        if i >= 6000:  # Only take first N
            break
        code = line.strip()
        game_codes.append(code)

In [ ]:
len(game_codes)

In [ ]:
model_path='qnet_mc_pretrained.pth'
# Load models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(model_path, map_location=device)

# Initialize networks
online_model = QNetwork(input_dim=138).to(device)
online_model.load_state_dict(checkpoint['model_state_dict'])

target_model = QNetwork(input_dim=138).to(device)
target_model.load_state_dict(checkpoint['model_state_dict'])

scaler = checkpoint['scaler']
feature_gen = FeatureGenerator()


In [ ]:
# Generate tuples with TD errors
print("Generating training tuples with TD errors...")
training_tuples = generate_training_tuples_with_td(
    # game_codes[:1000],  # Use first 1000 games
    game_codes,
    online_model,
    target_model,
    scaler,
    feature_gen,
    alpha=0.1,
    gamma=0.99
)

In [ ]:
len(training_tuples)

In [ ]:
training_tuples[1]

In [ ]:
replay_buffer = SmartReplayBuffer(capacity=100000)
for tuple_data in training_tuples:
    replay_buffer.push(tuple_data)

In [ ]:
replay_buffer.get_statistics()

In [ ]:
replay_buffer.plot_td_error_analysis()

In [ ]:
# Train with prioritized replay
print("\nTraining with prioritized replay...")
losses, td_history = train_with_smart_buffer(
    online_model,
    replay_buffer,
    scaler,
    epochs=100,
    batch_size=256,
    lr=1e-4,
    use_prioritized=True
)

In [ ]:
torch.save({
    'model_state_dict': online_model.state_dict(),
    'scaler': scaler,
    'losses': losses,
    'td_history': td_history
}, 'qnet_improved.pth')


In [ ]:
def complete_td_training_loop(starting_position=1e6, total_iterations=10):
    """
    Full TD learning with refreshing buffers
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load initial models
    checkpoint = torch.load("qnet_mc_pretrained.pth", map_location=device)
    online_model = QNetwork(input_dim=138).to(device)
    online_model.load_state_dict(checkpoint['model_state_dict'])

    target_model = QNetwork(input_dim=138).to(device)
    target_model.load_state_dict(checkpoint['model_state_dict'])

    scaler = checkpoint['scaler']
    feature_gen = FeatureGenerator()

    all_losses = []

    for iteration in range(total_iterations):
        print(f"\n{'='*50}")
        print(f"TD ITERATION {iteration+1}/{total_iterations}")
        print(f"{'='*50}")

        # 1. Load NEW games each iteration
        start_pos = int(starting_position + iteration * 10000)
        print(f"Loading games from position {start_pos}")

        game_codes = []
        codes_file = os.path.expanduser('~/Downloads/replayMem.txt')

        with open(codes_file, 'r') as f:
            for _ in range(start_pos):
                f.readline()
            for i, line in enumerate(f):
                if i >= 6000:
                    break
                game_codes.append(line.strip())

        # 2. Generate TD targets with CURRENT models
        print("Generating fresh TD targets...")
        training_tuples = generate_training_tuples_with_td(
            game_codes,
            online_model,    # Uses current online model!
            target_model,    # Uses current target model
            scaler,
            feature_gen,
            alpha=0.1,
            gamma=0.99
        )

        # 3. Create fresh buffer
        replay_buffer = SmartReplayBuffer(capacity=100000)
        for tuple_data in training_tuples:
            replay_buffer.push(tuple_data)

        stats = replay_buffer.get_statistics()
        print(f"Buffer stats: {len(replay_buffer.buffer)} samples")
        print(f"Avg TD error: {stats['avg_td_error']:.4f}")

        # 4. Train
        print("Training on fresh TD targets...")
        losses, td_hist = train_with_smart_buffer(
            online_model,
            replay_buffer,
            scaler,
            epochs=100,
            batch_size=256,
            lr=5e-5 * (0.9 ** iteration),  # Decay LR each iteration
            use_prioritized=True
        )

        all_losses.extend(losses)

        # 5. Update target network every 3 iterations
        if (iteration + 1) % 30 == 0:
            target_model.load_state_dict(online_model.state_dict())
            print("✓ Updated target network")

        # 6. Save checkpoint
        torch.save({
            'model_state_dict': online_model.state_dict(),  # The trained model
            'scaler': scaler,                               # For inference
            'iteration': iteration,                         # Track progress
        }, f'qnet_td_iter_{iteration+1}.pth')
        # 7. Optional: Quick test vs original
        if (iteration + 1) % 5 == 0:
            print("\nQuick performance check... which we are not doing")
            # Run 10 games vs original model
            # ... test code ...

    return online_model, all_losses

In [ ]:
complete_td_training_loop()

This is a test for why we are not getting zero errors when training with the alpha equal to 0 (which means TD targets are not taken into account)

In [ ]:
# Cell that disassembles full_training_pipeline
# This particular cell takes 6000 game codes and gets ready to produce TD difference cells.
codes_file = os.path.expanduser('~/Downloads/replayMem.txt')
skip_rows = 1e6
skip_rows = int(skip_rows)
game_codes = []
with open(codes_file, 'r') as f:
    print(f"Skipping a {skip_rows} rows")
    for _ in range(skip_rows):
        f.readline()
    for i, line in enumerate(f):
        if i >= 6000:  # Only take first N
            break
        code = line.strip()
        game_codes.append(code)

In [ ]:
len(game_codes)

In [ ]:
model_path='qnet_mc_pretrained.pth'
# Load models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(model_path, map_location=device)

# Initialize networks
online_model = QNetwork(input_dim=138).to(device)
online_model.load_state_dict(checkpoint['model_state_dict'])

target_model = QNetwork(input_dim=138).to(device)
target_model.load_state_dict(checkpoint['model_state_dict'])

scaler = checkpoint['scaler']
feature_gen = FeatureGenerator()


In [ ]:
# Verifying to check why the training is weird:
# Generate tuples with TD errors
print("Generating training tuples with TD errors...")
training_tuples = generate_training_tuples_with_td(
    game_codes[:100],  # Just 100 games for quick test
    online_model,
    target_model,
    scaler,
    feature_gen,
    alpha=0.0,  # <-- CHANGE TO 0
    gamma=0.99
)

In [ ]:
# Train with prioritized replay
print("\nTraining with prioritized replay...")
losses, td_history = train_with_smart_buffer(
    online_model,
    replay_buffer,
    scaler,
    epochs=1,  # <-- Just 1 epoch
    batch_size=256,
    lr=1e-4,
    use_prioritized=True
)
print(f"\nFirst loss with alpha=0: {losses[0]:.8f}")

In [ ]:
# Example usage combining everything - feed it game codes and you are done
# Let's split it into a cell to execute
def full_training_pipeline(game_codes, model_path='qnet_mc_pretrained.pth'):
    """
    Complete training pipeline with pre-computed TD errors
    """
    # Load models
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(model_path, map_location=device)

    # Initialize networks
    online_model = QNetwork(input_dim=138).to(device)
    online_model.load_state_dict(checkpoint['model_state_dict'])

    target_model = QNetwork(input_dim=138).to(device)
    target_model.load_state_dict(checkpoint['model_state_dict'])

    scaler = checkpoint['scaler']
    feature_gen = FeatureGenerator()

    # Generate tuples with TD errors
    print("Generating training tuples with TD errors...")
    training_tuples = generate_training_tuples_with_td(
        game_codes[:1000],  # Use first 1000 games
        online_model,
        target_model,
        scaler,
        feature_gen,
        alpha=0.1,
        gamma=0.99
    )

    # Fill replay buffer
    replay_buffer = SmartReplayBuffer(capacity=100000)
    for tuple_data in training_tuples:
        replay_buffer.push(tuple_data)

    print(f"Buffer filled with {len(replay_buffer.buffer)} tuples")
    stats = replay_buffer.get_statistics()
    print(f"Initial statistics: {stats}")

    # Train with prioritized replay
    print("\nTraining with prioritized replay...")
    losses, td_history = train_with_smart_buffer(
        online_model,
        replay_buffer,
        scaler,
        epochs=100,
        batch_size=256,
        lr=1e-4,
        use_prioritized=True
    )

    # Save the trained model
    torch.save({
        'model_state_dict': online_model.state_dict(),
        'scaler': scaler,
        'losses': losses,
        'td_history': td_history
    }, 'qnet_improved.pth')

    return online_model, losses, td_history


# Minimal version for testing - not sure if needed
def minimal_train_with_td(training_tuples, model, scaler, epochs=50):
    """
    Ultra-minimal training using pre-computed TD errors
    Sorts by TD error and trains on highest error samples
    """
    device = next(model.parameters()).device
    optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)

    # Sort by TD error (descending)
    sorted_tuples = sorted(training_tuples, key=lambda x: x.td_error, reverse=True)

    for epoch in range(epochs):
        # Focus on top 50% highest TD error samples
        high_error_samples = sorted_tuples[:len(sorted_tuples)//2]
        np.random.shuffle(high_error_samples)

        total_loss = 0
        for i in range(0, len(high_error_samples), 32):
            batch = high_error_samples[i:i+32]
            if not batch:
                continue

            states = torch.FloatTensor([t.state_action for t in batch]).to(device)
            targets = torch.FloatTensor([t.target_q for t in batch]).to(device)

            states_scaled = scaler.transform(states.cpu().numpy())
            states_scaled = torch.FloatTensor(states_scaled).to(device)

            preds = model(states_scaled).squeeze()
            loss = ((preds - targets) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            avg_td = np.mean([t.td_error for t in high_error_samples])
            print(f"Epoch {epoch}: Loss={total_loss:.6f}, AvgTD={avg_td:.4f}")